# LongerSession 承接关系 encoder 训练（完整版）

一次跑通：挂载 Drive → 下载数据 → 对比学习训练 → TIAGE 评测。

**使用步骤**：
1. 上传本 notebook 到 Colab，运行时选 **T4 GPU**
2. 菜单 → 运行时 → **全部运行**
3. 中途会弹 Google Drive 授权，点「允许」
4. 训练约 1 小时，模型自动存到 Drive 的 `longersession_model3`

> 训练期间保持 Colab 标签页打开，避免运行时断连。

In [ ]:
!pip install -q sentence-transformers datasets accelerate


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print('GPU available:', torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo 'NO GPU - 请切到 T4 运行时'

In [ ]:
# ③ 下载 OASST1 并构建正样本对
from datasets import load_dataset, concatenate_datasets
ds = load_dataset('OpenAssistant/oasst1')
all_ds = concatenate_datasets([ds['train'], ds['validation']])
df = all_ds.to_pandas()

m = df[(df['deleted']==False) & (df['review_result']==True) & (df['tree_state']=='ready_for_export') & (df['role'].isin(['prompter','assistant'])) & (df['lang'].isin(['en','zh']))].copy()
m['text'] = m['text'].astype(str).str.strip()
m = m[m['text'].str.len()>0]

by_text = dict(zip(m['message_id'], m['text']))
by_role = dict(zip(m['message_id'], m['role']))

pairs = []
for mid, pid, role, text in zip(m['message_id'], m['parent_id'], m['role'], m['text']):
    if pid in by_text:
        pr = by_role[pid]
        if pr == 'prompter' and role == 'assistant':
            pairs.append({'anchor': by_text[pid], 'positive': text})
        elif pr == 'assistant' and role == 'prompter':
            pairs.append({'anchor': by_text[pid], 'positive': text})
print('positive pairs:', len(pairs))

In [ ]:
# ④ 对比学习训练（cosine 衰减 + 5 epoch），存到 Drive
import os, shutil
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.losses import MultipleNegativesRankingLoss

# ===== 可调配置 =====
MODEL = "BAAI/bge-m3"   # 568M 多语言基座（Colab Pro L4/A100）
EPOCHS = 3
MAX_SEQ_LENGTH = 2048   # 关键：session 的 turn 可能很长，512 不够；可调 4096/8192
BATCH_SIZE = 8
GRAD_ACCUM = 4   # 梯度累积，等效 batch = 8*4 = 32
LR = 2e-5
OUTPUT_DIR = "/content/model_bgem3"
DRIVE_DIR = "/content/drive/MyDrive/longersession_model_bgem3"
# ====================

steps_per_epoch = len(pairs) // BATCH_SIZE
total_steps = (steps_per_epoch * EPOCHS) // GRAD_ACCUM   # 优化器步数
warmup_steps = int(total_steps * 0.1)
print(f'total_steps={total_steps} warmup_steps={warmup_steps}')

model = SentenceTransformer(MODEL)
model.max_seq_length = MAX_SEQ_LENGTH   # 长 turn 需要更长上下文
model.gradient_checkpointing_enable()    # 长序列省显存（换一点速度）
print('max_seq_length:', model.max_seq_length)
dataset = Dataset.from_list(pairs)
loss = MultipleNegativesRankingLoss(model, scale=20.0)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    fp16=torch.cuda.is_available(),
    bf16=False,
    logging_steps=100,
    save_strategy='epoch',
    report_to=[],
    dataloader_drop_last=True,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=dataset, loss=loss)
trainer.train()
model.save_pretrained(OUTPUT_DIR)

os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree(OUTPUT_DIR, DRIVE_DIR, dirs_exist_ok=True)
print('saved to Drive ->', DRIVE_DIR)

In [ ]:
# ⑤ TIAGE 评测
import requests, json, numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('/content/model_bgem3')

r = requests.get('https://api.github.com/repos/HuiyuanXie/tiage/git/trees/main?recursive=1')
paths = [t['path'] for t in r.json()['tree'] if 'anno_test.json' in t['path']]
url = 'https://raw.githubusercontent.com/HuiyuanXie/tiage/main/' + paths[0]
data = requests.get(url).json()

cases = []
for did, turns in data.items():
    utterances = [u for u, l in turns]
    boundaries = [i for i, (u, l) in enumerate(turns) if l == '1']
    cases.append((utterances, boundaries))

all_texts = []
offsets = []
for utts, _ in cases:
    offsets.append(len(all_texts))
    all_texts.extend(utts)

embs = model.encode(all_texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
case_scores = []
for idx, (utts, _) in enumerate(cases):
    s = offsets[idx]
    e = embs[s:s+len(utts)]
    case_scores.append([float(np.dot(e[i], e[i+1])) for i in range(len(e)-1)])

def dp_segment(scores, min_len=2, max_len=20, threshold=0.5):
    n = len(scores) + 1
    if n <= min_len:
        return []
    INF = float('inf')
    dp = [INF]*(n+1); prev = [-1]*(n+1); dp[0] = 0.0
    for i in range(1, n+1):
        lo = max(0, i-max_len); hi = i-min_len
        for j in range(lo, hi+1):
            if i-j < min_len or i-j > max_len:
                continue
            cost = dp[j] + ((scores[j-1]-threshold) if j > 0 else 0.0)
            if cost < dp[i]:
                dp[i] = cost; prev[i] = j
    if dp[n] == INF:
        return []
    cuts = []; i = n
    while i > 0:
        j = prev[i]
        if j <= 0:
            break
        cuts.append(j); i = j
    return sorted(cuts)

def evaluate_all(golds, preds, tolerance=0):
    tp = fp = fn = 0
    for g, p in zip(golds, preds):
        gs = set(g); matched = set(); t = 0
        for x in p:
            for y in gs:
                if y in matched:
                    continue
                if abs(x-y) <= tolerance:
                    t += 1; matched.add(y); break
        tp += t; fp += len(p)-t; fn += len(gs)-len(matched)
    precision = tp/(tp+fp) if tp+fp else 0.0
    recall = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
    return f1, precision, recall

golds = [b for _, b in cases]
print('threshold   F1(tol0)   F1(tol1)   P       R')
best = None
for th in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    preds = [dp_segment(s, threshold=th) for s in case_scores]
    f1, p, r = evaluate_all(golds, preds, 0)
    f1t, _, _ = evaluate_all(golds, preds, 1)
    print(f'{th:.2f}       {f1:.3f}       {f1t:.3f}       {p:.3f}   {r:.3f}')
    if best is None or f1 > best[1]:
        best = (th, f1, f1t, p, r)
print('=== best: threshold=%.2f F1(tol0)=%.3f F1(tol1)=%.3f ===' % (best[0], best[1], best[2]))
print('对照: word_bag 0.329 | bge-m3 零样本 0.397 | bge-small微调 0.429')